In [0]:
# Databricks notebook source

# UC3 - Part A: RAG Policy Q&A
# MediLife Insurance - primeinsurance_genai.gold.dim_policy
#
# This notebook:
#   1. Loads dim_policy from Unity Catalog
#   2. Converts each row to a rich, unambiguous natural-language document
#   3. Measures document lengths and sets chunk size dynamically from the data
#   4. Chunks and embeds documents using all-MiniLM-L6-v2 (local, no external API)
#   5. Builds a FAISS vector index
#   6. Answers natural-language questions by retrieving relevant chunks
#   7. Logs all queries to primeinsurance_genai.gold.rag_query_history

# COMMAND ----------

# Install required packages.
# sentence-transformers: local embedding model, no external API calls.
# faiss-cpu: exact vector similarity search.
%pip install sentence-transformers faiss-cpu --quiet
dbutils.library.restartPython()


In [0]:

# COMMAND ----------

import pandas as pd
import numpy as np
import uuid
import time
from datetime import datetime, timezone
from typing import List, Dict, Tuple

from sentence_transformers import SentenceTransformer
import faiss

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, FloatType, IntegerType, TimestampType, ArrayType
)

from openai import OpenAI

In [0]:

spark = SparkSession.builder.getOrCreate()

# COMMAND ----------

# Config - table names and model endpoint.
# CHUNK_SIZE and CHUNK_OVERLAP are set dynamically in Section 3
# after measuring actual document lengths from the data.

CATALOG       = "primeinsurance_genai"
SCHEMA        = "gold"
SOURCE_TABLE  = f"{CATALOG}.{SCHEMA}.dim_policy"
HISTORY_TABLE = f"{CATALOG}.{SCHEMA}.rag_query_history_v2"

EMBED_MODEL = "all-MiniLM-L6-v2"  # runs locally on the driver, no external API
TOP_K       = 5                    # number of chunks to retrieve per query

# Databricks model serving via OpenAI-compatible SDK.
# Notebook token authenticates to the workspace endpoint - no external key needed.
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
WORKSPACE_URL    = spark.conf.get("spark.databricks.workspaceUrl")

In [0]:

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"https://{WORKSPACE_URL}/serving-endpoints"
)

LLM_MODEL = "databricks-meta-llama-3-3-70b-instruct"

print("Config loaded.")
print(f"Source table : {SOURCE_TABLE}")
print(f"History table: {HISTORY_TABLE}")
print(f"LLM endpoint : {LLM_MODEL}")

In [0]:
# COMMAND ----------

# ── SECTION 1: Load dim_policy ───────────────────────────────────────────────
#
# Filter to is_current = true to skip stale SCD2 history rows.

policy_df = spark.table(SOURCE_TABLE).filter(F.col("is_current") == True)

print(f"Current policies loaded: {policy_df.count():,}")
policy_df.printSchema()
policy_df.show(3, truncate=False)

In [0]:

# COMMAND ----------

# ── SECTION 2: Structured row -> natural-language document ───────────────────
#
# Why convert a structured table to prose?
#   Embedding models encode semantic meaning from text. Raw values like
#   6000000 or "100/300" have no semantic relationship to query phrases like
#   "umbrella coverage" or "property damage claim" in the model's vector space.
#   Prose conversion creates that relationship.
#
# Key design decisions in this version vs a naive conversion:
#
#   1. UMBRELLA: Old format used "Umbrella coverage: NO umbrella coverage" and
#      "Umbrella coverage: YES - limit $X" — the word "umbrella" appears in both,
#      so the embedding model places YES and NO documents near each other in vector
#      space and cannot separate them. New format uses "HAS umbrella coverage" vs
#      "does NOT have umbrella coverage" giving the model a clear directional signal.
#
#   2. CSL: Old format said "Bodily injury and property damage limits" which implies
#      two separate fields. The LLM kept debating which field applied to property
#      damage claims. New format explicitly states "combined single limit (CSL)"
#      and "covers both bodily injury and property damage" so the LLM receives
#      this definition in the retrieved chunk itself.
#
#   3. SUFFICIENCY LABELS: Adding "can cover property damage claims up to $X"
#      directly in the document means coverage scenario queries retrieve the right
#      chunks via semantic match rather than relying on the model to do arithmetic.
#
#   4. DEDUCTIBLE TIERS: Adding "(low/standard/high deductible tier)" means
#      comparative queries retrieve chunks with matching tier language.
#
#   5. STATE ABBREVIATION: Repeating both "Illinois" and "IL" means queries
#      using either form retrieve correctly.

def row_to_natural_language(row: dict) -> str:
    """
    Serialise a single dim_policy row into a rich prose document.
    Every chunk produced from this document is fully self-contained
    and unambiguous for semantic retrieval.
    """
    pol      = str(row.get("policy_number", "Unknown"))
    bind     = str(row.get("policy_bind_date", ""))[:10]
    state    = str(row.get("policy_state_full", "Unknown"))
    state_cd = str(row.get("policy_state_code", ""))
    csl      = str(row.get("policy_csl", "N/A"))
    ded      = int(row.get("policy_deductable", 0))
    prem     = float(row.get("policy_annual_premium", 0.0))

    # Parse CSL into explicit per-person and per-occurrence values.
    # Label the per-occurrence limit with a sufficiency statement so
    # coverage scenario queries ("covers $450K damage") retrieve correctly.
    csl_parts = csl.split("/")
    if len(csl_parts) == 2:
        per_person     = int(csl_parts[0])
        per_occurrence = int(csl_parts[1])
        csl_text = (
            f"The combined single limit (CSL) is {csl}, meaning ${per_person}K per person "
            f"and ${per_occurrence}K per occurrence. This single limit covers both bodily "
            f"injury and property damage claims."
        )
        if per_occurrence >= 1000:
            csl_text += " This policy can cover large property damage claims up to $1,000,000."
        elif per_occurrence >= 500:
            csl_text += " This policy can cover property damage claims up to $500,000."
        else:
            csl_text += " This policy covers property damage claims up to $300,000."
    else:
        csl_text = f"Coverage limit: {csl}."

    # Deductible with explicit tier label for comparative queries
    if ded <= 500:
        ded_text = f"The deductible is ${ded:,} (low deductible tier)."
    elif ded <= 1000:
        ded_text = f"The deductible is ${ded:,} (standard deductible tier)."
    else:
        ded_text = f"The deductible is ${ded:,} (high deductible tier)."

    # Umbrella with explicit positive/negative language.
    # "HAS umbrella" vs "does NOT have umbrella" gives the embedding model
    # a clear directional signal so YES and NO documents land in different
    # regions of vector space.
    umbrella_flag = row.get("_umbrella_limit_zero_flag", True)
    umbrella_val  = float(row.get("umbrella_limit", 0))

    if not umbrella_flag and umbrella_val > 0:
        umbrella_text = (
            f"This policy HAS umbrella coverage with an additional limit of "
            f"${umbrella_val:,.0f}. Umbrella coverage provides extra liability "
            f"protection beyond the base CSL limit."
        )
    else:
        umbrella_text = (
            f"This policy does NOT have umbrella coverage. "
            f"No additional liability protection beyond the base CSL limit."
        )

    doc = (
        f"Policy {pol} is an active auto insurance policy bound on {bind}, "
        f"issued in {state} ({state_cd}). "
        f"{csl_text} "
        f"{ded_text} "
        f"The annual premium is ${prem:,.2f}. "
        f"{umbrella_text} "
        f"Car ID: {row.get('car_id', 'N/A')}. "
        f"Customer ID: {row.get('customer_id', 'N/A')}."
    )
    return doc


# Convert all current rows to prose documents
policy_pd = policy_df.toPandas()
policy_pd["nl_document"] = policy_pd.apply(
    lambda r: row_to_natural_language(r.to_dict()), axis=1
)

print(f"Documents generated: {len(policy_pd):,}")

# Spot-check that YES and NO umbrella documents look semantically distinct
print("\n── Umbrella YES example ─────────────────────────────────────────────")
sample_yes = policy_pd[policy_pd["_umbrella_limit_zero_flag"] == False].iloc[0]
print(sample_yes["nl_document"])

print("\n── Umbrella NO example ──────────────────────────────────────────────")
sample_no = policy_pd[policy_pd["_umbrella_limit_zero_flag"] == True].iloc[0]
print(sample_no["nl_document"])

In [0]:
# COMMAND ----------

# ── SECTION 3: Measure document lengths and set chunk size dynamically ────────
#
# The new richer documents are longer than the old format.
# Old format: ~270-320 characters per policy.
# New format: ~480-580 characters per policy.
#
# At the old CHUNK_SIZE=300 a new document would split mid-sentence,
# separating the policy number (start of doc) from the umbrella section
# (end of doc). The second chunk would have no policy number anchor
# and could not be cited correctly.
#
# We measure the actual P95 document length from the data and set
# CHUNK_SIZE = P95 + buffer so that 95% of policies produce exactly
# one chunk. This keeps the one-policy-per-chunk principle intact.

policy_pd["doc_length"] = policy_pd["nl_document"].apply(len)

len_min  = policy_pd["doc_length"].min()
len_max  = policy_pd["doc_length"].max()
len_mean = policy_pd["doc_length"].mean()
len_p95  = policy_pd["doc_length"].quantile(0.95)

print("── Document length distribution ─────────────────────────────────────")
print(f"  Min  : {len_min}")
print(f"  Max  : {len_max}")
print(f"  Mean : {len_mean:.0f}")
print(f"  P95  : {len_p95:.0f}")

# Set chunk size dynamically from the data.
# Adding 50 chars above P95 gives a comfortable buffer for the longest rows
# without making chunks so large that retrieval becomes imprecise.
CHUNK_SIZE    = int(len_p95) + 50
CHUNK_OVERLAP = 75  # increased from 50 proportionally for longer documents.
                    # If a doc does split, overlap carries the policy number
                    # from chunk 0 into chunk 1 so it remains citable.

print(f"\n── Chunk size set from data ─────────────────────────────────────────")
print(f"  CHUNK_SIZE    = P95 ({int(len_p95)}) + 50 buffer = {CHUNK_SIZE}")
print(f"  CHUNK_OVERLAP = {CHUNK_OVERLAP}")

# Show how many documents will split at this chunk size.
# Target is near zero - if this number is high, increase the buffer.
splits = policy_pd[policy_pd["doc_length"] > CHUNK_SIZE]
print(f"\n  Documents that will split at CHUNK_SIZE={CHUNK_SIZE}: "
      f"{len(splits)} of {len(policy_pd)} "
      f"({100 * len(splits) / len(policy_pd):.1f}%)")

if len(splits) > 0:
    print(f"  Longest document: {len_max} chars")
    print(f"  Consider increasing buffer if split % is above 5%")

In [0]:
# COMMAND ----------

# ── SECTION 4: Chunking ───────────────────────────────────────────────────────
#
# One policy per chunk is the goal.
# CHUNK_SIZE was set from the data in Section 3 so this is now satisfied
# for 95%+ of policies without any manual tuning.

def chunk_document(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    """
    Sliding-window character chunker with overlap.
    For documents shorter than chunk_size (most policies), returns a single chunk.
    For longer documents, produces overlapping chunks so no field is orphaned.
    """
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start  = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start += chunk_size - overlap
    return chunks


# Build chunk corpus with policy metadata attached to each chunk.
# Metadata is stored in a parallel list (chunk_meta) where position i
# corresponds exactly to row i in the FAISS index.
chunk_records = []
for _, row in policy_pd.iterrows():
    doc   = row["nl_document"]
    pol   = str(row["policy_number"])
    state = str(row.get("policy_state_full", row.get("policy_state_code", "")))

    for i, chunk in enumerate(chunk_document(doc)):
        chunk_records.append({
            "chunk_id":      f"{pol}_c{i}",
            "policy_number": pol,
            "state":         state,
            "chunk_text":    chunk,
        })

chunks_df = pd.DataFrame(chunk_records)

print(f"Total chunks    : {len(chunks_df):,}")
print(f"Total policies  : {len(policy_pd):,}")
print(f"Avg chunks/policy: {len(chunks_df) / len(policy_pd):.3f}")

# If avg chunks per policy is close to 1.000 the chunk size is correct.
# If it is significantly above 1.0 the documents are splitting too often.
if len(chunks_df) / len(policy_pd) > 1.05:
    print("WARNING: avg chunks per policy > 1.05 - consider increasing CHUNK_SIZE")
else:
    print("OK: avg chunks per policy is near 1.0 - chunk size is well calibrated")

print(f"\nSample chunk:\n{chunks_df['chunk_text'].iloc[0]}")

In [0]:

# COMMAND ----------

# ── SECTION 5: Local embeddings with all-MiniLM-L6-v2 ────────────────────────
#
# The model runs entirely on the Databricks driver node.
# No external API is called for embedding generation.
# normalize_embeddings=True scales each vector to unit length so that
# inner-product search (IndexFlatIP) equals cosine similarity.

print(f"Loading embedding model: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)
print("Model loaded.")

print(f"\nEmbedding {len(chunks_df):,} chunks ...")
t0 = time.time()

embeddings = embed_model.encode(
    chunks_df["chunk_text"].tolist(),
    batch_size=256,
    show_progress_bar=True,
    normalize_embeddings=True,
)

elapsed = time.time() - t0
print(f"Done. shape={embeddings.shape}  time={elapsed:.1f}s")

In [0]:

# COMMAND ----------

# ── SECTION 6: Build FAISS index ──────────────────────────────────────────────
#
# IndexFlatIP performs exact inner-product search.
# With L2-normalised vectors, inner product equals cosine similarity.
# The dataset (~965 policies) is small enough for exact search in milliseconds.

dim   = embeddings.shape[1]  # 384 for all-MiniLM-L6-v2
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype("float32"))

# Parallel metadata list - position i matches FAISS index row i
chunk_meta = chunks_df.to_dict(orient="records")

print(f"FAISS index: {index.ntotal:,} vectors, dim={dim}")
print(f"Metadata   : {len(chunk_meta):,} entries")

# COMMAND ----------

# ── SECTION 7: Retrieval and answer pipeline ──────────────────────────────────

import re

def retrieve(query: str, top_k: int = TOP_K) -> Tuple[List[Dict], np.ndarray]:
    """
    Retrieve top-k chunks for a query using two strategies:

    Strategy 1 - Exact policy number match:
      If the query contains a numeric policy number that exists in the metadata,
      that policy's chunk is pinned to position 0 with a score of 1.0.
      This guarantees specific-policy lookup queries always cite the correct record
      regardless of where FAISS ranks it. Policy number lookup is a keyword
      problem, not a semantic similarity problem.

    Strategy 2 - Semantic search:
      FAISS cosine similarity over all chunk embeddings fills the remaining slots.
      The richer document format (HAS/does NOT have umbrella, explicit sufficiency
      labels, deductible tier labels) means semantically relevant chunks now score
      clearly higher than irrelevant ones without needing separate sub-indexes.
    """
    # Check if the query contains a policy number that exists in our data
    numbers_in_query = re.findall(r'\b\d{5,7}\b', query)
    exact_match = None

    for num in numbers_in_query:
        for meta in chunk_meta:
            if meta["policy_number"] == num:
                exact_match = meta.copy()
                exact_match["score"] = 1.0
                break
        if exact_match:
            break

    # Run FAISS semantic search
    q_vec = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_vec, top_k)
    scores  = scores[0]
    indices = indices[0]

    results = []
    for score, idx in zip(scores, indices):
        if idx == -1:
            continue
        meta = chunk_meta[idx].copy()
        meta["score"] = float(score)
        # Skip the exact match policy from semantic results to avoid duplication
        if exact_match and meta["policy_number"] == exact_match["policy_number"]:
            continue
        results.append(meta)

    # Pin exact match to position 0, fill remaining slots from semantic results
    if exact_match:
        results       = [exact_match] + results[:top_k - 1]
        return_scores = np.array([1.0] + list(scores[:top_k - 1]))
    else:
        results       = results[:top_k]
        return_scores = scores[:top_k]

    return results, return_scores

In [0]:
def build_prompt(query: str, chunks: List[Dict]) -> str:
    """
    Assemble the LLM prompt with retrieved chunks as numbered sources.

    The CSL definition is included in the system instruction so the LLM
    never debates whether bodily injury and property damage are separate fields.
    Each source is labelled with its policy number and cosine score so the
    LLM has explicit citation targets.
    """
    context_parts = []
    for i, c in enumerate(chunks, 1):
        context_parts.append(
            f"[SOURCE {i} | Policy {c['policy_number']} | score={c['score']:.3f}]\n"
            f"{c['chunk_text']}"
        )
    context = "\n\n".join(context_parts)

    prompt = (
        "You are a claims adjuster assistant for MediLife Insurance.\n"
        "Answer the question below using ONLY the policy context provided.\n"
        "Always cite specific policy numbers (e.g. Policy 100804) in your answer.\n"
        "If the context does not contain enough information, say so clearly.\n\n"
        "IMPORTANT: The combined single limit (CSL) field covers both bodily injury "
        "and property damage under one limit. The format X/Y means $XK per person "
        "bodily injury and $YK per occurrence total. The per-occurrence limit is the "
        "relevant figure for property damage claims.\n\n"
        "--- POLICY CONTEXT ---\n"
        f"{context}\n\n"
        "--- QUESTION ---\n"
        f"{query}\n\n"
        "--- ANSWER (cite policy numbers) ---"
    )
    return prompt


def compute_confidence(scores: np.ndarray) -> float:
    """
    Heuristic confidence based on the average of the top-3 cosine scores.
    all-MiniLM-L6-v2 scores range roughly 0.30 (random) to 0.95 (near-duplicate).
    Normalise to 0-1 and clamp.
    """
    top3 = scores[:3]
    raw  = float(np.mean(top3))
    conf = (raw - 0.30) / (0.95 - 0.30)
    return round(min(max(conf, 0.0), 1.0), 4)


def ask(query: str, top_k: int = TOP_K) -> Dict:
    """
    End-to-end RAG pipeline:
      1. Retrieve top-k chunks (exact match pinned + semantic search)
      2. Build prompt with retrieved chunks as grounded context
      3. Call Databricks LLM via OpenAI-compatible endpoint
      4. Return structured result dict ready to log
    """
    chunks, scores  = retrieve(query, top_k=top_k)
    source_policies = list(dict.fromkeys(c["policy_number"] for c in chunks))

    prompt = build_prompt(query, chunks)

    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=512,
        )
        answer = response.choices[0].message.content.strip()
    except Exception as e:
        print(f"LLM call failed: {e}")
        answer = (
            f"Retrieved {len(chunks)} policy records. "
            f"Top match: {chunks[0]['chunk_text'] if chunks else 'No results.'}"
        )

    confidence = compute_confidence(scores)

    return {
        "query_id":         str(uuid.uuid4()),
        "question":         query,
        "answer":           answer,
        "confidence":       confidence,
        "source_policies":  source_policies,
        "chunks_retrieved": len(chunks),
        "retrieved_at":     datetime.now(timezone.utc),
    }


print("RAG pipeline ready.")

In [0]:
# COMMAND ----------

# ── SECTION 8: Create rag_query_history if it does not exist ──────────────────

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {HISTORY_TABLE} (
        query_id         STRING         COMMENT 'Unique UUID for this query',
        question         STRING         COMMENT 'Raw natural-language question from adjuster',
        answer           STRING         COMMENT 'LLM-generated answer with policy citations',
        confidence       FLOAT          COMMENT 'Heuristic confidence 0-1 based on cosine similarity',
        source_policies  ARRAY<STRING>  COMMENT 'Policy numbers cited in the answer',
        chunks_retrieved INT            COMMENT 'Number of FAISS chunks retrieved',
        retrieved_at     TIMESTAMP      COMMENT 'UTC timestamp of query'
    )
    USING DELTA
    COMMENT 'RAG query log for MediLife Policy Q&A system'
""")

print(f"Table ready: {HISTORY_TABLE}")

In [0]:
# COMMAND ----------

# ── SECTION 9: Run 5 test questions ───────────────────────────────────────────
#
# Question types covered:
#   Q1 - specific policy lookup (exact match + semantic)
#   Q2 - filter-based (umbrella YES vs NO - fixed by richer document language)
#   Q3 - comparative (deductible tiers - improved by tier labels in documents)
#   Q4 - state filter with feature check
#   Q5 - coverage scenario (sufficiency labels in documents fix retrieval)

# Q1 - Specific policy lookup
q1 = ask("What are the deductible and umbrella coverage details for policy 100804?")
print("Q1:", q1["question"])
print("Sources:", q1["source_policies"])
print("Confidence:", q1["confidence"])
print("Answer:", q1["answer"])
print()

# COMMAND ----------

# Q2 - Filter: policies with umbrella coverage
# The richer document language (HAS vs does NOT have) means the retriever
# now surfaces umbrella-positive policies without needing a sub-index.
q2 = ask("Which policies have umbrella coverage, and what are their umbrella limits?")
print("Q2:", q2["question"])
print("Sources:", q2["source_policies"])
print("Confidence:", q2["confidence"])
print("Answer:", q2["answer"])
print()

# COMMAND ----------

# Q3 - Comparative: deductible tiers
# Explicit tier labels in the document (low/standard/high deductible tier)
# improve retrieval for this query type.
q3 = ask(
    "Compare policies with a $500 deductible versus a $2000 deductible "
    "- which is more common and what premiums do they carry?"
)
print("Q3:", q3["question"])
print("Sources:", q3["source_policies"])
print("Confidence:", q3["confidence"])
print("Answer:", q3["answer"])
print()

# COMMAND ----------

# Q4 - State filter with feature check
q4 = ask(
    "What bodily injury limits are typical for Illinois policies, "
    "and do any of them have umbrella coverage?"
)
print("Q4:", q4["question"])
print("Sources:", q4["source_policies"])
print("Confidence:", q4["confidence"])
print("Answer:", q4["answer"])
print()

# COMMAND ----------

# Q5 - Coverage scenario
# Explicit sufficiency labels in the document ("can cover property damage
# claims up to $X") mean the retriever surfaces policies with sufficient
# limits without the LLM needing to interpret raw CSL numbers.
q5 = ask(
    "A claimant says the insured hit a fence and caused $450,000 in property damage. "
    "Which retrieved policies would have sufficient limits to cover this?"
)
print("Q5:", q5["question"])
print("Sources:", q5["source_policies"])
print("Confidence:", q5["confidence"])
print("Answer:", q5["answer"])
print()

In [0]:
# COMMAND ----------

# ── SECTION 10: Log all 5 queries to rag_query_history ────────────────────────

results = [q1, q2, q3, q4, q5]

log_rows = [
    (
        r["query_id"],
        r["question"],
        r["answer"],
        float(r["confidence"]),
        r["source_policies"],
        r["chunks_retrieved"],
        r["retrieved_at"],
    )
    for r in results
]

log_schema = StructType([
    StructField("query_id",         StringType(),            False),
    StructField("question",         StringType(),            False),
    StructField("answer",           StringType(),            True),
    StructField("confidence",       FloatType(),             True),
    StructField("source_policies",  ArrayType(StringType()), True),
    StructField("chunks_retrieved", IntegerType(),           True),
    StructField("retrieved_at",     TimestampType(),         True),
])

log_sdf = spark.createDataFrame(log_rows, schema=log_schema)
log_sdf.write.format("delta").mode("append").saveAsTable(HISTORY_TABLE)

print(f"{len(results)} queries logged to {HISTORY_TABLE}")


In [0]:

# COMMAND ----------

# ── SECTION 11: Verify - read back latest 5 rows only ─────────────────────────
#
# Deduplicates by question, keeping only the most recent run per question.
# This avoids accumulating duplicate rows across multiple notebook runs.

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

w = Window.partitionBy("question").orderBy(desc("retrieved_at"))

display(
    spark.table(HISTORY_TABLE)
    .withColumn("rn", row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select(
        "query_id",
        "question",
        "answer",
        "confidence",
        "source_policies",
        "chunks_retrieved",
        "retrieved_at"
    )
)